# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   
<br> Url: https://github.com/JavierRodriguezReina/03MIAR-Proyecto-Grupo16.git <br>
Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

Los datos son:

- Número de actores 10
- Número de tomas 30
- Valores (nuestro github): [Enlace a los datos](https://github.com/JavierRodriguezReina/03MIAR-Proyecto-Grupo16/blob/main/Datos%20problema%20doblaje(30%20tomas%2C%2010%20actores).csv)

> 1 significa que el actor participa en la toma

> 0 significa que el actor no participa en la toma
                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

**Respuesta**

**Modelo del espacio de soluciones.** Una solución consiste en decidir **en qué día se graba cada toma**. La representamos, igual que en el modelo de tupla del árbol de expansión (p.ej. las N reinas), como una tupla de 30 posiciones:

$$(d_1, d_2, \dots, d_{30}), \qquad d_i = \text{día en que se graba la toma } i$$

donde cada componente puede tomar como valor cualquiera de los días disponibles. Como cada una de las 30 tomas puede ir a cualquier día, esto es una **variación con repetición** (combinatoria básica).

### Sin tener en cuenta las restricciones

Si hay $D$ días posibles, el número de tuplas distintas es:

$$VR_D^{30} = D^{30}$$

En el caso extremo cada toma va a su propio día, luego como máximo hacen falta **30 días** ($D = 30$):

$$D^{30} = 30^{30} \approx 2{,}06 \times 10^{44}$$

Este es el tamaño del árbol de expansión que recorrería la fuerza bruta (días etiquetados, sin descartar equivalentes ni días vacíos), y lo reutilizaremos al calcular su complejidad.

> **Nota:** si en lugar de tuplas contamos únicamente las **agrupaciones distintas** (sin etiquetar los días, sin repetir las equivalentes), el problema equivale a las *particiones de un conjunto* de 30 elementos, cuyo número es el **número de Bell** $B(30) \approx 8{,}47 \times 10^{23}$. Es un valor menor porque no sobrecuenta las ordenaciones de días equivalentes.

### Teniendo en cuenta todas las restricciones

La restricción del problema es **máximo 6 tomas por día**. Entonces ya no vale cualquier tupla: solo son válidas aquellas en las que **ningún valor de día aparece más de 6 veces** (ningún día concentra más de 6 tomas). Esto reduce el espacio a un subconjunto mucho menor del anterior, que calculamos por conteo en la siguiente celda.

In [8]:
from functools import lru_cache
from math import comb

N_TOMAS = 30      # número de tomas
MAX_DIA = 6       # máximo de tomas por día (restricción)

# ----- SIN restricciones: variaciones con repetición (enfoque del temario) -----
# Cada toma se asigna a uno de los D días posibles -> D^N.
# En el caso extremo cada toma va a su propio día, así que D = N (30 días).
D = N_TOMAS
sin_restriccion = D ** N_TOMAS

# ----- Número de Bell B(N): agrupaciones DISTINTAS, sin etiquetar los días -----
# (particiones de un conjunto de N elementos). Se genera con el triángulo de Bell.
def bell(n):
    fila = [1]
    for _ in range(n):
        nueva = [fila[-1]]                 # cada fila empieza con el último de la anterior
        for x in fila:
            nueva.append(nueva[-1] + x)    # cada término = izquierda + el de arriba
        fila = nueva
    return fila[0]

bell_30 = bell(N_TOMAS)

# ----- CON restricciones (<= 6 tomas por día) -----
# Agrupaciones (particiones) donde ningún grupo/día supera MAX_DIA tomas.
# Fijamos una toma: su día tendrá tamaño s (1..MAX_DIA); elegimos sus s-1 compañeros
# de entre las N-1 restantes y particionamos el resto.
@lru_cache(None)
def particiones_restringidas(n, k=MAX_DIA):
    if n == 0:
        return 1
    return sum(comb(n - 1, s - 1) * particiones_restringidas(n - s, k)
               for s in range(1, min(k, n) + 1))

con_restriccion = particiones_restringidas(N_TOMAS, MAX_DIA)

# ----- Comparación -----
print(f"{'Enfoque':45}{'Valor':>12}   Cuenta")
print("-" * 90)
print(f"{'SIN restr. · variaciones con repetición D^N':45}{sin_restriccion:>12.3e}   {sin_restriccion}")
print(f"{'SIN restr. · número de Bell B(30)':45}{bell_30:>12.3e}   {bell_30}")
print(f"{'CON restr. · Bell restringido (<=6/día)':45}{con_restriccion:>12.3e}   {con_restriccion}")
print("-" * 90)
print(f"D^N es {sin_restriccion / bell_30:,.0f} veces mayor que B(30) "
      f"(sobrecuenta ordenaciones de días equivalentes y días vacíos).")
print(f"El Bell restringido es el {con_restriccion / bell_30:.1%} de B(30) "
      f"(la restricción <=6 elimina solo las particiones con grupos grandes).")

Enfoque                                             Valor   Cuenta
------------------------------------------------------------------------------------------
SIN restr. · variaciones con repetición D^N     2.059e+44   205891132094649000000000000000000000000000000
SIN restr. · número de Bell B(30)               8.467e+23   846749014511809332450147
CON restr. · Bell restringido (<=6/día)         7.264e+23   726391948970868949621309
------------------------------------------------------------------------------------------
D^N es 243,154,852,932,843,307,008 veces mayor que B(30) (sobrecuenta ordenaciones de días equivalentes y días vacíos).
El Bell restringido es el 85.8% de B(30) (la restricción <=6 elimina solo las particiones con grupos grandes).


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

Una solución del problema consiste en **repartir las 30 tomas en días de grabación**. La cuestión es cómo representar ese reparto.

### Primera idea: una lista de conjuntos

Lo más intuitivo es guardar la solución como una **lista de días**, donde cada día es el **conjunto de tomas** que se graban en él:

$$[\;\{t_1, t_2, t_6\},\;\; \{t_3, t_5\},\;\; \{t_4\}\;]$$

Es una representación fiel: se lee directamente qué se graba cada día. Pero al pensar en cómo **recorrer el espacio de soluciones** aparecen dos problemas serios:

- **Longitud variable:** el número de días no es fijo, así que las soluciones no tienen todas la misma forma.
- **Invariante frágil:** cada toma debe aparecer **exactamente una vez** en toda la estructura. Cualquier operación de búsqueda (mover una toma, y sobre todo el **cruce** de un genético) rompe esa condición con facilidad y genera soluciones inválidas, con tomas duplicadas o perdidas, que habría que reparar a mano.

### Estructura elegida: una tupla de 30 enteros

Por eso cambiamos a una representación **posicional**: una tupla con una posición por toma, cuyo valor es el día que se le asigna.

$$(d_1, d_2, \dots, d_{30}), \qquad d_i = \text{día en que se graba la toma } i$$

Esta estructura resuelve los dos problemas anteriores y se adapta mucho mejor a la búsqueda:

- **Longitud fija (30 posiciones):** todas las soluciones tienen la misma forma.
- **Siempre válida por construcción:** cada toma ocupa una única posición, así que es imposible duplicarla o perderla. No hay nada que reparar.
- **Operadores triviales:** cambiar una decisión es cambiar un valor. En el **algoritmo genético** esta tupla es directamente el **cromosoma**: mutar es cambiar un gen y cruzar es partir el vector por un punto.
- **Permite contar el espacio de forma inmediata:** al ser una asignación posicional, el número de soluciones sale como variaciones con repetición, $D^{30}$ (pregunta anterior).

**Las dos representaciones son equivalentes** y se pasa de una a otra agrupando las tomas que comparten día (*decodificar*). Usamos la tupla para **buscar** y la agrupación en días solo cuando hace falta **evaluar** el coste de la solución:

$$\underbrace{(0,\,0,\,1,\,2,\,1,\,0)}_{\text{tupla: se busca}} \;\xrightarrow{\;\text{decodificar}\;}\; \underbrace{[\,\{t_1,t_2,t_6\},\;\{t_3,t_5\},\;\{t_4\}\,]}_{\text{días: se evalúa}}$$

**Conclusión:** el espacio de soluciones se modela con una **tupla de 30 enteros**. Aunque la lista de conjuntos es más legible, la tupla es la que mejor se adapta al problema porque tiene longitud fija, es válida por construcción y permite aplicar los operadores de búsqueda (cruce y mutación) sin generar soluciones inconsistentes.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

### La función objetivo

Los actores cobran **por cada día que se desplazan al estudio**, todos la misma cantidad e independientemente de cuántas tomas se graben ese día. Por tanto el gasto no depende del número de tomas, sino de **cuántos actores acuden cada día**, sumado sobre todos los días. La unidad natural del coste es el **actor·día**.

Dada una solución $d = (d_1, \dots, d_{30})$, el coste de un día $j$ es el número de **actores distintos** que necesitan las tomas asignadas a ese día, es decir, el tamaño de la **unión** de sus conjuntos de actores. La función objetivo es la suma sobre todos los días:

$$f(d) \;=\; \sum_{j \,\in\, \text{días}} \; \Bigl|\; \bigcup_{i \;:\; d_i = j} \text{actores}(i) \;\Bigr|$$

donde $\text{actores}(i)$ es el conjunto de actores que participan en la toma $i$.

El gasto real en euros sería $f(d) \times$ (tarifa diaria por actor). Como la tarifa es la misma para todos los actores, minimizar el gasto equivale a minimizar $f(d)$.

**Restricciones:**

- Cada toma se graba exactamente un día. Queda garantizado por la propia tupla: cada toma ocupa una única posición.
- Ningún día puede tener más de **6 tomas**: $\;|\{\, i : d_i = j \,\}| \le 6\;$ para todo día $j$.

### ¿Maximización o minimización?

Es un problema de **minimización**. El enunciado pide *"que el gasto por los servicios de los actores de doblaje sea el menor posible"*, y $f(d)$ es exactamente ese gasto. Buscamos, entre todas las soluciones que cumplen las restricciones, la que haga $f(d)$ mínima:

$$\min_{d} \; f(d) \qquad \text{sujeto a} \qquad |\{\, i : d_i = j \,\}| \le 6 \quad \forall j$$

**Intuición:** agrupar en un mismo día las tomas que comparten actores hace que esos actores acudan una sola vez en lugar de varias. Un actor que participa en 5 tomas repartidas en 5 días distintos cuesta 5 actor·día; si esas 5 tomas se graban el mismo día, cuesta 1. Minimizar $f$ consiste, por tanto, en **concentrar las tomas de cada actor en el menor número de días posible**.

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

**Respuesta**

### Idea

La fuerza bruta consiste en **generar todas las soluciones posibles**, evaluar cada una con la función objetivo y quedarse con la de menor coste. Al ser exhaustiva, **garantiza el óptimo**: si existiera una solución mejor, la habría visitado.

### Diseño del algoritmo

Recorremos las tomas una a una y, para cada toma, probamos **todos los días a los que puede ir**. Esto genera el árbol de expansión completo del espacio de soluciones y se implementa de forma natural con **vuelta atrás** (recorrido en profundidad), donde cada nivel del árbol fija una posición de la tupla $(d_1, \dots, d_{30})$:

```
asignar(i):                                  # i = toma que toca colocar
    si i == 30:                              # ya están todas las tomas asignadas
        coste = f(solucion)                  # evaluar con la función objetivo
        si coste < mejor_coste:
            guardar solucion como la mejor
        return

    para cada día d posible:
        si el día d tiene menos de 6 tomas:  # <- restricción del problema
            solucion[i] = d                  # asignar la toma i al día d
            asignar(i + 1)                   # continuar con la siguiente toma
            deshacer la asignación           # vuelta atrás
```

### Un detalle importante: no repetir soluciones

Si dejamos que cada toma vaya a cualquiera de los $D$ días, generamos $D^{30}$ tuplas, pero **muchas son la misma solución con los días renombrados**: `(0,0,1)` y `(1,1,0)` describen exactamente la misma agrupación. Para no repetir trabajo aplicamos una regla sencilla al elegir el día de cada toma:

> la toma $i$ solo puede ir a un **día ya abierto**, o **abrir el siguiente día nuevo** (nunca saltarse números de día).

Así cada agrupación distinta se genera **exactamente una vez**, y el número de soluciones que recorre el algoritmo es justamente el que calculamos en la primera pregunta: las agrupaciones con **≤ 6 tomas por día**, $\approx 7{,}26 \times 10^{23}$, en lugar de $30^{30} \approx 2{,}06 \times 10^{44}$.

Sigue siendo **fuerza bruta**: no hay ninguna poda por cotas ni heurística que descarte ramas, se exploran **todas** las soluciones. Simplemente no se generan duplicados.

### El problema: no es ejecutable para 30 tomas

El algoritmo es correcto, pero **inaplicable al caso real**. En la celda siguiente lo ejecutamos sobre subconjuntos de tomas cada vez mayores y se observa la explosión combinatoria:

| Nº de tomas | Soluciones exploradas | Tiempo |
|---|---|---|
| 4 | 15 | 0,0001 s |
| 6 | 203 | 0,0008 s |
| 8 | 4.131 | 0,014 s |
| 10 | 115.274 | 0,39 s |
| 11 | 672.673 | 2,4 s |
| 12 | 4.163.743 | 15,6 s |

Cada toma adicional **multiplica** el tiempo (de 11 a 12 tomas se multiplica por más de 6). A la velocidad medida ($\approx 2{,}8 \times 10^5$ soluciones/s), completar las **30 tomas** exigiría:

$$\frac{7{,}26 \times 10^{23}\ \text{soluciones}}{2{,}8 \times 10^{5}\ \text{soluciones/s}} \;\approx\; 2{,}6 \times 10^{18}\ \text{s} \;\approx\; \mathbf{8{,}1 \times 10^{10}\ \text{años}}$$

Es decir, unas **6 veces la edad del universo**.

**Conclusión:** la fuerza bruta solo es útil aquí como **referencia de corrección** (da el óptimo garantizado en instancias pequeñas, con el que validar otros algoritmos). Para el problema real es imprescindible un algoritmo que **no explore todo el espacio de soluciones**.

In [11]:
import csv
import time
from math import inf

RUTA_CSV = "Datos problema doblaje(30 tomas, 10 actores).csv"
NUM_ACTORES = 10
MAX_TOMAS_DIA = 6      # restricción: no más de 6 tomas por día


# ===== Datos: cada toma -> conjunto de actores que participan en ella =====
tomas = []
with open(RUTA_CSV, newline="") as f:
    for fila in csv.reader(f):
        if fila and fila[0].strip().isdigit():                     # filas de tomas
            valores = [int(v) for v in fila[1:1 + NUM_ACTORES]]
            tomas.append({a + 1 for a, v in enumerate(valores) if v == 1})


# ===== Modelo y función objetivo =====
def decodificar(solucion):
    """Agrupa las tomas por el día que les asigna la tupla. Devuelve {día: [índices de toma]}."""
    dias = {}
    for i_toma, dia in enumerate(solucion):
        dias.setdefault(dia, []).append(i_toma)
    return dias


def funcion_objetivo(solucion, lista_tomas=None):
    """f(d) = suma, para cada día, de los actores DISTINTOS que deben acudir. A MINIMIZAR."""
    lista_tomas = tomas if lista_tomas is None else lista_tomas
    return sum(len(set().union(*(lista_tomas[i] for i in indices)))
               for indices in decodificar(solucion).values())


def es_valida(solucion):
    """Comprueba la restricción: ningún día con más de 6 tomas."""
    return all(len(idx) <= MAX_TOMAS_DIA for idx in decodificar(solucion).values())


# ===== Algoritmo por fuerza bruta =====
def fuerza_bruta(lista_tomas, max_tomas_dia=MAX_TOMAS_DIA):
    """
    Explora TODAS las agrupaciones posibles de las tomas en días (con <= 6 tomas/día)
    y devuelve la de menor coste. Garantiza el óptimo.

    Para no generar la misma solución varias veces con los días renombrados, cada toma
    solo puede ir a un día ya abierto o abrir el siguiente día nuevo.
    """
    n = len(lista_tomas)
    solucion = [0] * n              # tupla en construcción: solucion[i] = día de la toma i
    ocupacion = [0] * n             # nº de tomas ya asignadas a cada día
    mejor = {"coste": inf, "solucion": None}
    exploradas = 0

    def asignar(i, dias_abiertos):
        nonlocal exploradas
        if i == n:                                   # solución completa -> evaluarla
            exploradas += 1
            c = funcion_objetivo(solucion, lista_tomas)
            if c < mejor["coste"]:
                mejor["coste"] = c
                mejor["solucion"] = tuple(solucion)
            return

        # la toma i va a un día ya abierto (0..dias_abiertos-1) o abre el día 'dias_abiertos'
        for dia in range(dias_abiertos + 1):
            if ocupacion[dia] < max_tomas_dia:       # restricción: máx. 6 tomas por día
                solucion[i] = dia
                ocupacion[dia] += 1
                asignar(i + 1, max(dias_abiertos, dia + 1))
                ocupacion[dia] -= 1                  # vuelta atrás

    asignar(0, 0)
    return mejor["coste"], mejor["solucion"], exploradas


# ===== Ejecución sobre subconjuntos crecientes: la explosión combinatoria =====
print(f"Tomas cargadas: {len(tomas)}   (ejemplo -> Toma 1: {tomas[0]})\n")
print(f"{'nº tomas':>9}{'soluciones':>14}{'mejor coste':>13}{'tiempo (s)':>13}")
print("-" * 49)
for n in [4, 6, 8, 10, 11]:
    t0 = time.time()
    mejor_coste, mejor_sol, exploradas = fuerza_bruta(tomas[:n])
    t = time.time() - t0
    print(f"{n:>9}{exploradas:>14,}{mejor_coste:>13}{t:>13.4f}")
    velocidad = exploradas / t

print(f"\nMejor solución con {n} tomas: {mejor_sol}  ->  coste {mejor_coste} actor·día")

# ===== Extrapolación al problema real (30 tomas) =====
SOLUCIONES_30 = 726_391_948_970_868_949_621_309     # calculado en la pregunta 1
segundos = SOLUCIONES_30 / velocidad
anios = segundos / (3600 * 24 * 365)
print(f"\nVelocidad medida: {velocidad:,.0f} soluciones/s")
print(f"Soluciones a explorar con 30 tomas: {SOLUCIONES_30:.3e}")
print(f"Tiempo estimado: {segundos:.3e} s  =  {anios:.3e} años")
print(f"-> Unas {anios / 13.8e9:.0f} veces la edad del universo. INVIABLE.")


FileNotFoundError: [Errno 2] No such file or directory: 'Datos problema doblaje(30 tomas, 10 actores).csv'

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

El coste de la fuerza bruta es el producto de dos factores:

$$\text{Complejidad} \;=\; \underbrace{(\text{nº de soluciones que se generan})}_{\text{recorrer el espacio}} \;\times\; \underbrace{(\text{coste de evaluar una solución})}_{\text{función objetivo}}$$

### 1. Coste de evaluar una solución

Para calcular $f(d)$ hay que recorrer las $n$ tomas y unir sus conjuntos de actores. Cada conjunto tiene como mucho $m$ actores ($m = 10$), así que la unión total cuesta:

$$O(n \cdot m)$$

### 2. Número de soluciones generadas

**Versión ingenua (asignación libre):** cada una de las $n$ tomas puede ir a cualquiera de los $D$ días, luego se generan $D^{n}$ tuplas (variaciones con repetición). En el peor caso $D = n$:

$$O(D^{n}) \;\xrightarrow{\;D = n\;}\; O(n^{n})$$

**Versión implementada (sin duplicados):** al obligar a que cada toma vaya a un día ya abierto o abra el siguiente, se genera cada agrupación **una sola vez**. El número de soluciones es entonces el de **particiones del conjunto de tomas con bloques de tamaño ≤ 6**, acotado por el **número de Bell** $B(n)$:

$$O(B(n)), \qquad B(n) \sim \left(\frac{n}{\ln n}\right)^{n}$$

### Complejidad total

$$\boxed{\;T(n) \;=\; O\bigl(B(n) \cdot n \cdot m\bigr)\;}$$

y en la versión ingenua, $O(n^{n} \cdot n \cdot m)$.

En ambos casos el factor dominante es el número de soluciones, que crece de forma **superexponencial**: más rápido que $2^{n}$ (y que cualquier exponencial $c^{n}$), aunque más lento que $n!$.

$$2^{n} \;\ll\; B(n) \;\ll\; n!$$

### Comprobación empírica

Si el algoritmo fuese $O(2^n)$, al añadir una toma el tiempo se multiplicaría **siempre por 2**. Lo que medimos es que el factor **no es constante y va creciendo**:

| $n$ | Soluciones | Factor respecto a $n-1$ |
|---|---|---|
| 10 | 115.274 | — |
| 11 | 672.673 | × 5,8 |
| 12 | 4.163.743 | × 6,2 |

Ese factor creciente es precisamente la firma de un crecimiento **superexponencial**, y confirma la cota $O(B(n))$.

### Consecuencia práctica

Para $n = 30$ el algoritmo debe explorar $\approx 7{,}26 \times 10^{23}$ soluciones. A la velocidad medida ($\approx 2{,}8 \times 10^{5}$ soluciones/s) tardaría unos $8 \times 10^{10}$ años. **La fuerza bruta es intratable**: no basta con un ordenador más rápido, porque el problema crece más deprisa de lo que puede compensar el hardware. Hace falta un algoritmo que **no recorra todo el espacio de soluciones**.

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta



### Por qué la fuerza bruta no sirve

La fuerza bruta falla porque **recorre el espacio de soluciones entero**: $O(B(n))$, superexponencial. La única forma de mejorarla es renunciar a explorarlo todo, y eso implica renunciar también a la **garantía de optimalidad**.

Ese es el intercambio: la fuerza bruta da el óptimo garantizado pero tarda $10^{10}$ años; un algoritmo **aproximado** da una solución muy buena en segundos. Diseñamos un **algoritmo genético**, una metaheurística que explora el espacio de forma guiada en lugar de exhaustiva.

### El algoritmo genético

La idea es imitar la evolución natural: se mantiene una **población** de soluciones que compiten entre sí, y las mejores se reproducen y se refinan generación tras generación.

**Codificación.** Cada individuo es un **cromosoma** = la tupla $(d_1, \dots, d_{30})$ del modelo, donde $d_i$ es el día asignado a la toma $i$. Aquí se ve por qué elegimos la tupla: cualquier vector de enteros es una solución válida, así que el cruce y la mutación nunca producen individuos corruptos.

**Función de *fitness*.** Es la función objetivo más una **penalización** por incumplir la restricción:

$$\text{fitness}(d) \;=\; f(d) \;+\; 100 \cdot \sum_{j} \max\bigl(0,\; |\{i : d_i = j\}| - 6\bigr)$$

Si un día tiene más de 6 tomas se castiga con 100 puntos por cada toma sobrante. Así las soluciones que violan la restricción resultan carísimas y el algoritmo las descarta por sí solo, sin necesidad de repararlas. Como es un problema de **minimización**, se busca el fitness más bajo.

**Ciclo evolutivo.** En cada generación:

1. **Selección por torneo:** se cogen 3 individuos al azar y gana el de mejor fitness. Se repite para elegir los dos padres.
2. **Cruce de un punto:** se parten los cromosomas de los padres por una posición aleatoria y se combinan los trozos.
3. **Mutación:** cada gen tiene un 4 % de probabilidad de cambiar a un día aleatorio. Introduce diversidad y evita quedar atrapado en óptimos locales.
4. **Elitismo:** los 2 mejores individuos pasan intactos a la siguiente generación, para no perder nunca la mejor solución encontrada.

**Parámetros:** población 300, 800 generaciones, 6 días disponibles, mutación 0,04.

### Por qué mejora la fuerza bruta

La diferencia es **estructural**, no una cuestión de ir más rápido:

| | Fuerza bruta | Genético |
|---|---|---|
| Soluciones que evalúa | **todas**: $B(30) \approx 7{,}3 \times 10^{23}$ | **las que decidimos**: $5 \times 300 \times 800 = 1{,}2$ millones |
| ¿Depende de $n$? | sí, **superexponencialmente** | **no**: lo fijamos nosotros |
| Tiempo (30 tomas) | $\approx 8 \times 10^{10}$ años | **≈ 15 segundos** |
| ¿Óptimo garantizado? | sí | no, pero solución muy buena |

La clave está en la segunda fila: **el número de soluciones que evalúa el genético no depende del tamaño del problema**, sino de los parámetros que elegimos (población × generaciones). El coste deja de explotar con $n$.

Y no explora al azar: la **selección** hace que las buenas soluciones se reproduzcan más, el **cruce** combina fragmentos útiles de dos padres y la **mutación** evita el estancamiento. Por eso encuentra soluciones muy buenas evaluando solo una fracción infinitesimal del espacio: **1,2 millones de soluciones de $7{,}3 \times 10^{23}$**, un $1{,}7 \times 10^{-16}$ % del total.

Como el genético es **estocástico** (parte de una población aleatoria), hacemos **5 reinicios** y nos quedamos con el mejor resultado, que es la práctica habitual con metaheurísticas.

### Validación: ¿es fiable renunciar al óptimo garantizado?

La objeción evidente al genético es que **no garantiza el óptimo**. Para comprobar si eso es un problema real, lo contrastamos con la fuerza bruta en instancias **pequeñas**, donde esta última sí es ejecutable y nos da el óptimo verdadero:

| Nº de tomas | Óptimo (fuerza bruta) | Genético | ¿Coincide? |
|---|---|---|---|
| 6 | 7 | 7 | ✅ |
| 8 | 12 | 12 | ✅ |
| 10 | 13 | 13 | ✅ |
| 11 | 13 | 13 | ✅ |
| 12 | 14 | 14 | ✅ |

**El genético encuentra el óptimo exacto en todos los casos en los que podemos verificarlo.** Eso da confianza en la solución que produce para las 30 tomas, donde ya no hay forma de comprobarlo.

### Resultado sobre el problema real (30 tomas)

| Algoritmo | Coste (actor·día) | Tiempo |
|---|---|---|
| Fuerza bruta | óptimo garantizado | $\approx 8 \times 10^{10}$ años (**inviable**) |
| **Genético** | **28** | ≈ 15 s (5 reinicios) |
| *Cota inferior teórica* | *21* | — |

> **Cota inferior:** un actor que participa en $k$ tomas debe acudir al menos $\lceil k/6 \rceil$ días, porque no caben más de 6 tomas diarias. Sumando para los 10 actores salen 21 actor·día. Ninguna solución puede bajar de ahí, así que el óptimo real está entre **21 y 28**.

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta